In [1]:
!pip install nltk spacy word2number
!python -m spacy download en_core_web_sm
%pip uninstall -y torchvision torchaudio torchtext
%pip install -q -U "transformers>=4.48.0" "accelerate>=0.26.0" datasets scikit-learn pandas nltk seaborn matplotlib


  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=a723f64547f513adbb77f70c53fed559d6fb059526d28c0251bdb858f614f955
  Stored in directory: /root/.cache/pip/wheels/5b/79/fb/d25928e599c7e11fe4e00d32048cd74933f34a74c633d2aea6
Successfully built word2number
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 63.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfu

In [2]:
# --- Hybrid ModernBERT + ELECTRA WITHOUT Augmentation ---
import os
import random
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

os.environ["WANDB_DISABLED"] = "true"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DATA_URL = 'https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv'
df = pd.read_csv(DATA_URL)
df = df[['RequirementText', 'class']].dropna().reset_index(drop=True)
df['model_text'] = df['RequirementText'].astype(str)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])
train_df, val_test_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_used_df = train_df.copy()
print("Train distribution (no augmentation):", dict(train_used_df['class'].value_counts()))
print("Validation distribution:", dict(val_df['class'].value_counts()))
print("Test distribution:", dict(test_df['class'].value_counts()))

mb_tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
el_tok = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128, return_token_type_ids=False)

class HybridDataset(Dataset):
    def __init__(self, texts, labels):
        self.mb = tokenize(mb_tok, texts)
        self.el = tokenize(el_tok, texts)
        self.labels = [int(label) for label in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "mb_input_ids": torch.tensor(self.mb["input_ids"][idx], dtype=torch.long),
            "mb_attention_mask": torch.tensor(self.mb["attention_mask"][idx], dtype=torch.long),
            "el_input_ids": torch.tensor(self.el["input_ids"][idx], dtype=torch.long),
            "el_attention_mask": torch.tensor(self.el["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = HybridDataset(train_used_df["model_text"].astype(str).tolist(), train_used_df["label"].tolist())
val_ds = HybridDataset(val_df["model_text"].astype(str).tolist(), val_df["label"].tolist())
test_ds = HybridDataset(test_df["model_text"].astype(str).tolist(), test_df["label"].tolist())


def freeze_lower_encoder_layers(encoder):
    for layer_path, layers in [
        ("encoder.layer", getattr(getattr(encoder, "encoder", None), "layer", None)),
        ("layers", getattr(encoder, "layers", None)),
        ("model.layers", getattr(getattr(encoder, "model", None), "layers", None)),
    ]:
        if layers is not None and len(layers) >= 8:
            # Freeze lower transformer layers to reduce overfitting and computation on short requirement texts.
            for layer in layers[:8]:
                for param in layer.parameters():
                    param.requires_grad = False
            print(f"Frozen lower transformer layers: {layer_path}[0:8]")
            return
    raise ValueError("Could not find transformer layers to freeze.")


def report_trainable_and_frozen_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    frozen_params = sum(param.numel() for param in model.parameters() if not param.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {frozen_params:,}")

class GatedFusion(nn.Module):
    def __init__(self, n_labels, weights=None):
        super().__init__()
        self.mb = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
        self.el = AutoModel.from_pretrained("google/electra-base-discriminator")
        freeze_lower_encoder_layers(self.mb)
        freeze_lower_encoder_layers(self.el)
        self.gate = nn.Linear(768 * 2, 768)
        self.dropout = nn.Dropout(0.3)
        self.clf = nn.Linear(768, n_labels)
        report_trainable_and_frozen_parameters(self)
        if weights is not None:
            self.register_buffer("weights", weights)
        else:
            self.weights = None

    def forward(self, mb_input_ids=None, mb_attention_mask=None, el_input_ids=None, el_attention_mask=None, labels=None, **kwargs):
        mb_outputs = self.mb(input_ids=mb_input_ids, attention_mask=mb_attention_mask)
        el_outputs = self.el(input_ids=el_input_ids, attention_mask=el_attention_mask)
        h_mb = mb_outputs.last_hidden_state[:, 0, :]
        h_el = el_outputs.last_hidden_state[:, 0, :]
        h_concat = torch.cat([h_mb, h_el], dim=-1)
        g = torch.sigmoid(self.gate(h_concat))
        h_fused = g * h_mb + (1 - g) * h_el
        logits = self.clf(self.dropout(h_fused))
        loss = F.cross_entropy(logits, labels, weight=self.weights) if labels is not None else None
        return {"loss": loss, "logits": logits}

num_labels = len(label_encoder.classes_)
weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_labels), y=train_used_df["label"].astype(int))
weights = torch.tensor(weights, dtype=torch.float)
model = GatedFusion(n_labels=num_labels, weights=weights)

for param in model.parameters():
    if not param.is_contiguous():
        param.data = param.data.contiguous()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": p_w,
        "recall": r_w,
        "f1": f1_w,
        "precision_macro": p_m,
        "recall_macro": r_m,
        "macro_F1": f1_m,
    }

training_args = TrainingArguments(
    output_dir="./modernbert_electra_gated_fusion_" + ("with_augmentation" if False else "without_augmentation"),
    num_train_epochs=30,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_F1",
    greater_is_better=True,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

print("Validation Results:")
print(trainer.evaluate(val_ds))
print("\nTesting Results:")
test_results = trainer.predict(test_ds)
test_preds = np.argmax(test_results.predictions, axis=-1)
print(classification_report(test_df["label"].astype(int).tolist(), test_preds, target_names=label_encoder.classes_, zero_division=0))


Train distribution (no augmentation): {'F': np.int64(311), 'SE': np.int64(87), 'US': np.int64(59), 'O': np.int64(54), 'PE': np.int64(47), 'LF': np.int64(34), 'A': np.int64(22), 'MN': np.int64(17), 'SC': np.int64(15), 'FT': np.int64(13), 'L': np.int64(11), 'PO': np.int64(8)}
Validation distribution: {'F': np.int64(66), 'SE': np.int64(19), 'US': np.int64(13), 'O': np.int64(11), 'PE': np.int64(10), 'LF': np.int64(7), 'MN': np.int64(4), 'A': np.int64(4), 'SC': np.int64(4), 'FT': np.int64(3), 'PO': np.int64(2), 'L': np.int64(2)}
Test distribution: {'F': np.int64(67), 'SE': np.int64(19), 'US': np.int64(13), 'O': np.int64(12), 'PE': np.int64(10), 'LF': np.int64(8), 'A': np.int64(5), 'SC': np.int64(3), 'MN': np.int64(3), 'FT': np.int64(2), 'L': np.int64(2), 'PO': np.int64(2)}


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,Macro F1
1,16.755884,1.860150,0.455172,0.207182,0.455172,0.284752,0.037931,0.083333,0.052133
2,13.885645,1.668609,0.468966,0.303168,0.468966,0.331839,0.106767,0.100376,0.081954
3,12.013678,1.432872,0.579310,0.455595,0.579310,0.503402,0.180084,0.216128,0.190782
4,9.266125,1.236530,0.648276,0.578758,0.648276,0.600266,0.305172,0.313561,0.283865
5,7.876067,1.087520,0.696552,0.646538,0.696552,0.656288,0.420647,0.381172,0.368813
6,5.173151,1.015298,0.682759,0.641379,0.682759,0.649175,0.456566,0.432308,0.418803
7,3.363068,0.913964,0.724138,0.686002,0.724138,0.701340,0.484821,0.481944,0.479311
8,2.042429,0.890000,0.744828,0.708347,0.744828,0.722977,0.500105,0.522258,0.507597
9,0.926735,0.900021,0.751724,0.729522,0.751724,0.735929,0.505460,0.577246,0.535439
10,0.871518,0.973718,0.731034,0.724591,0.731034,0.720475,0.601308,0.560137,0.559520


Validation Results:


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,Macro F1
0.001717,1.190120,30,0.751724,0.761620,0.751724,0.746108,0.687806,0.624531,0.634045


{'eval_loss': 1.1901203393936157, 'eval_accuracy': 0.7517241379310344, 'eval_precision': 0.761619914318565, 'eval_recall': 0.7517241379310344, 'eval_f1': 0.7461076604554865, 'eval_precision_macro': 0.6878055519359867, 'eval_recall_macro': 0.6245308784782468, 'eval_macro_F1': 0.634044551798175}

Testing Results:


              precision    recall  f1-score   support

           A       1.00      0.40      0.57         5
           F       0.78      0.94      0.85        67
          FT       0.00      0.00      0.00         2
           L       0.67      1.00      0.80         2
          LF       0.50      0.25      0.33         8
          MN       0.00      0.00      0.00         3
           O       0.58      0.58      0.58        12
          PE       0.64      0.90      0.75        10
          PO       0.00      0.00      0.00         2
          SC       0.00      0.00      0.00         3
          SE       0.70      0.74      0.72        19
          US       0.80      0.62      0.70        13

    accuracy                           0.73       146
   macro avg       0.47      0.45      0.44       146
weighted avg       0.68      0.73      0.69       146



In [3]:
# --- Hybrid ModernBERT + ELECTRA WITH Augmentation ---
import os
import random
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import nltk
from nltk.corpus import stopwords, wordnet

os.environ["WANDB_DISABLED"] = "true"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
KEEP_WORDS = {'not', 'no', 'shall', 'must', 'should', 'will', 'cannot', 'never', 'only'}
stop_words = set(stopwords.words('english')) - KEEP_WORDS

def get_synonyms(word):
    clean_word = re.sub(r'[^a-zA-Z]', '', str(word).lower())
    if not clean_word:
        return []
    synonyms = set()
    for syn in wordnet.synsets(clean_word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != clean_word:
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=1):
    words = str(text).split()
    new_words = words.copy()
    random_word_list = list(set([
        word for word in words
        if re.sub(r'[^a-zA-Z]', '', word).lower() not in stop_words and len(word) > 3
    ]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = str(text).split()
    if not words:
        return str(text)
    for _ in range(n):
        words.insert(random.randint(0, len(words) - 1), random.choice(phrases))
    return ' '.join(words)

def random_deletion(text, p=0.1):
    words = str(text).split()
    if len(words) <= 1:
        return str(text)
    remaining = [word for word in words if random.random() > p]
    return ' '.join(remaining) if remaining else random.choice(words)

def random_swap(text, n=1):
    words = str(text).split()
    if len(words) < 2:
        return str(text)
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return ' '.join(words)

def augment_text(text):
    aug_text = synonym_replacement(text, 1)
    aug_text = phrase_insertion(aug_text, 1)
    aug_text = random_swap(aug_text, 1)
    aug_text = random_deletion(aug_text, p=0.1)
    return ' '.join(str(aug_text).split())

DATA_URL = 'https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv'
df = pd.read_csv(DATA_URL)
df = df[['RequirementText', 'class']].dropna().reset_index(drop=True)
df['model_text'] = df['RequirementText'].astype(str)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])
train_df, val_test_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

augmentation_targets = {
    "FT": 200,
    "L": 200,
    "MN": 200,
    "A": 200,
    "LF": 200,
    "SC": 200,
    "PO": 200,
    "PE": 200,
    "O": 200,
    "US": 200,
    "SE": 100,
}
augmented = []
for cls, target in augmentation_targets.items():
    cls_rows = train_df[train_df['class'] == cls]
    needed = max(0, target - len(cls_rows))
    if len(cls_rows) == 0 or needed == 0:
        continue
    for _ in range(needed):
        row = cls_rows.sample(n=1, random_state=random.randint(0, 10**9)).iloc[0]
        aug_text = augment_text(row['RequirementText'])
        augmented.append({'model_text': aug_text, 'RequirementText': aug_text, 'class': cls, 'label': row['label']})
train_used_df = pd.concat([train_df, pd.DataFrame(augmented)], ignore_index=True) if len(augmented) > 0 else train_df.copy()
train_used_df = train_used_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Original train distribution:", dict(train_df['class'].value_counts()))
print("Augmented train distribution:", dict(train_used_df['class'].value_counts()))
print("Validation distribution:", dict(val_df['class'].value_counts()))
print("Test distribution:", dict(test_df['class'].value_counts()))

mb_tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
el_tok = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128, return_token_type_ids=False)

class HybridDataset(Dataset):
    def __init__(self, texts, labels):
        self.mb = tokenize(mb_tok, texts)
        self.el = tokenize(el_tok, texts)
        self.labels = [int(label) for label in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "mb_input_ids": torch.tensor(self.mb["input_ids"][idx], dtype=torch.long),
            "mb_attention_mask": torch.tensor(self.mb["attention_mask"][idx], dtype=torch.long),
            "el_input_ids": torch.tensor(self.el["input_ids"][idx], dtype=torch.long),
            "el_attention_mask": torch.tensor(self.el["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = HybridDataset(train_used_df["model_text"].astype(str).tolist(), train_used_df["label"].tolist())
val_ds = HybridDataset(val_df["model_text"].astype(str).tolist(), val_df["label"].tolist())
test_ds = HybridDataset(test_df["model_text"].astype(str).tolist(), test_df["label"].tolist())


def freeze_lower_encoder_layers(encoder):
    for layer_path, layers in [
        ("encoder.layer", getattr(getattr(encoder, "encoder", None), "layer", None)),
        ("layers", getattr(encoder, "layers", None)),
        ("model.layers", getattr(getattr(encoder, "model", None), "layers", None)),
    ]:
        if layers is not None and len(layers) >= 8:
            # Freeze lower transformer layers to reduce overfitting and computation on short requirement texts.
            for layer in layers[:8]:
                for param in layer.parameters():
                    param.requires_grad = False
            print(f"Frozen lower transformer layers: {layer_path}[0:8]")
            return
    raise ValueError("Could not find transformer layers to freeze.")


def report_trainable_and_frozen_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    frozen_params = sum(param.numel() for param in model.parameters() if not param.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {frozen_params:,}")

class GatedFusion(nn.Module):
    def __init__(self, n_labels, weights=None):
        super().__init__()
        self.mb = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
        self.el = AutoModel.from_pretrained("google/electra-base-discriminator")
        freeze_lower_encoder_layers(self.mb)
        freeze_lower_encoder_layers(self.el)
        self.gate = nn.Linear(768 * 2, 768)
        self.dropout = nn.Dropout(0.3)
        self.clf = nn.Linear(768, n_labels)
        report_trainable_and_frozen_parameters(self)
        if weights is not None:
            self.register_buffer("weights", weights)
        else:
            self.weights = None

    def forward(self, mb_input_ids=None, mb_attention_mask=None, el_input_ids=None, el_attention_mask=None, labels=None, **kwargs):
        mb_outputs = self.mb(input_ids=mb_input_ids, attention_mask=mb_attention_mask)
        el_outputs = self.el(input_ids=el_input_ids, attention_mask=el_attention_mask)
        h_mb = mb_outputs.last_hidden_state[:, 0, :]
        h_el = el_outputs.last_hidden_state[:, 0, :]
        h_concat = torch.cat([h_mb, h_el], dim=-1)
        g = torch.sigmoid(self.gate(h_concat))
        h_fused = g * h_mb + (1 - g) * h_el
        logits = self.clf(self.dropout(h_fused))
        loss = F.cross_entropy(logits, labels, weight=self.weights) if labels is not None else None
        return {"loss": loss, "logits": logits}

num_labels = len(label_encoder.classes_)
weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_labels), y=train_used_df["label"].astype(int))
weights = torch.tensor(weights, dtype=torch.float)
model = GatedFusion(n_labels=num_labels, weights=weights)

for param in model.parameters():
    if not param.is_contiguous():
        param.data = param.data.contiguous()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": p_w,
        "recall": r_w,
        "f1": f1_w,
        "precision_macro": p_m,
        "recall_macro": r_m,
        "macro_F1": f1_m,
    }

training_args = TrainingArguments(
    output_dir="./modernbert_electra_gated_fusion_" + ("with_augmentation" if True else "without_augmentation"),
    num_train_epochs=30,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_F1",
    greater_is_better=True,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

print("Validation Results:")
print(trainer.evaluate(val_ds))
print("\nTesting Results:")
test_results = trainer.predict(test_ds)
test_preds = np.argmax(test_results.predictions, axis=-1)
print(classification_report(test_df["label"].astype(int).tolist(), test_preds, target_names=label_encoder.classes_, zero_division=0))


Original train distribution: {'F': np.int64(311), 'SE': np.int64(87), 'US': np.int64(59), 'O': np.int64(54), 'PE': np.int64(47), 'LF': np.int64(34), 'A': np.int64(22), 'MN': np.int64(17), 'SC': np.int64(15), 'FT': np.int64(13), 'L': np.int64(11), 'PO': np.int64(8)}
Augmented train distribution: {'F': np.int64(311), 'PE': np.int64(200), 'US': np.int64(200), 'FT': np.int64(200), 'O': np.int64(200), 'SC': np.int64(200), 'LF': np.int64(200), 'A': np.int64(200), 'PO': np.int64(200), 'MN': np.int64(200), 'L': np.int64(200), 'SE': np.int64(100)}
Validation distribution: {'F': np.int64(66), 'SE': np.int64(19), 'US': np.int64(13), 'O': np.int64(11), 'PE': np.int64(10), 'LF': np.int64(7), 'MN': np.int64(4), 'A': np.int64(4), 'SC': np.int64(4), 'FT': np.int64(3), 'PO': np.int64(2), 'L': np.int64(2)}
Test distribution: {'F': np.int64(67), 'SE': np.int64(19), 'US': np.int64(13), 'O': np.int64(12), 'PE': np.int64(10), 'LF': np.int64(8), 'A': np.int64(5), 'SC': np.int64(3), 'MN': np.int64(3), 'FT': n

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,Macro F1
1,18.913770,2.076202,0.462069,0.325123,0.462069,0.311738,0.163690,0.096814,0.078852
2,14.410327,1.571427,0.489655,0.547321,0.489655,0.419705,0.478266,0.266228,0.285391
3,7.312697,1.003999,0.724138,0.708936,0.724138,0.701431,0.643335,0.559314,0.571524
4,3.061735,0.802826,0.731034,0.710485,0.731034,0.712671,0.590386,0.559354,0.564098
5,1.130085,0.728221,0.793103,0.773019,0.793103,0.776979,0.648585,0.613384,0.625101
6,0.402746,0.929267,0.772414,0.773234,0.772414,0.764325,0.610600,0.643603,0.611613
7,0.380279,1.266954,0.737931,0.767931,0.737931,0.745325,0.623849,0.652282,0.628768
8,0.109089,1.191826,0.772414,0.770423,0.772414,0.765811,0.648189,0.671262,0.651611
9,0.000916,1.192431,0.793103,0.791432,0.793103,0.786299,0.673822,0.701117,0.667019
10,0.000514,1.199967,0.806897,0.786126,0.806897,0.785952,0.654727,0.642972,0.639552


Validation Results:


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,Macro F1
0.000045,1.270085,30,0.813793,0.812561,0.813793,0.805904,0.747270,0.682211,0.695529


{'eval_loss': 1.2700848579406738, 'eval_accuracy': 0.8137931034482758, 'eval_precision': 0.8125613590985131, 'eval_recall': 0.8137931034482758, 'eval_f1': 0.8059036236374574, 'eval_precision_macro': 0.7472696448048562, 'eval_recall_macro': 0.682210698658067, 'eval_macro_F1': 0.6955293235839095}

Testing Results:


              precision    recall  f1-score   support

           A       1.00      0.60      0.75         5
           F       0.83      0.97      0.90        67
          FT       1.00      1.00      1.00         2
           L       0.67      1.00      0.80         2
          LF       0.83      0.62      0.71         8
          MN       0.50      0.33      0.40         3
           O       0.58      0.58      0.58        12
          PE       0.90      0.90      0.90        10
          PO       0.00      0.00      0.00         2
          SC       0.00      0.00      0.00         3
          SE       0.90      0.95      0.92        19
          US       0.90      0.69      0.78        13

    accuracy                           0.83       146
   macro avg       0.68      0.64      0.65       146
weighted avg       0.80      0.83      0.81       146

